# DiffusionBlocks++ on Colab: train once, compare Euler and Heun

This notebook trains a **DiffusionBlocks (`dblock`)** classifier on a Colab GPU, copies its checkpoint to Google Drive, then evaluates that same checkpoint with Euler and Heun. The training loss does not depend on the inference sampler, so you do not need to train a separate model for each method.

Set **Runtime → Change runtime type → GPU** before starting. The default starter run uses 5 `num_epochs`; for `dblock`, the training script multiplies this by `num_blocks` (3 by default), so that is 15 Lightning trainer epochs. Increase it for a more meaningful accuracy comparison.

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select a GPU runtime, then rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0))

## Connect Drive and get the code

The checkpoint will be copied to Drive so it survives the temporary Colab runtime.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content
!test -d /content/DiffusionBlocksPlusPlus || git clone --branch heun-method https://github.com/nhford/DiffusionBlocksPlusPlus.git
%cd /content/DiffusionBlocksPlusPlus

## Install dependencies

Colab already provides CUDA-enabled PyTorch. This installs the project's remaining packages and skips `flash-attn`, which this code does not directly use.

In [ ]:
%pip install 'lightning==2.5.0' 'wandb>=0.22.3' 'transformers==4.52.4' 'datasets==3.5.0' 'huggingface-hub[hf-xet]>=0.31.2' 'hf-transfer>=0.1.9' 'scipy>=1.16.2'
%env USE_TF=0

## Train the model

This creates `logs/` automatically. W&B logging is online in this version, so training metrics appear in your W&B run.

## Sign in to Weights & Biases

Run this cell and complete the W&B authentication prompt before starting training. The training command below runs online (without `--debug`) so metrics are uploaded as training progresses.

In [ ]:
import wandb
wandb.login()

In [ ]:
EPOCHS = 5
BATCH_SIZE = 128
NUM_BLOCKS = 3

!python main.py train cifar100 --model_type dblock \
  --num_blocks {NUM_BLOCKS} --num_epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} --num_workers 2

## Copy a saved checkpoint to Drive

Review the checkpoint paths, then the next cell selects the most recently modified `last.ckpt` from a dblock run and copies it to Drive. If you prefer a best-validation checkpoint, select the corresponding `*.ckpt` path shown above and change `SOURCE` below.

In [ ]:
!find logs -type f -name '*.ckpt' -print

In [ ]:
from pathlib import Path
import shutil

candidates = sorted(
    Path("logs").glob("*dblock*/last.ckpt"),
    key=lambda p: p.stat().st_mtime,
)
if not candidates:
    raise FileNotFoundError("No dblock last.ckpt found under logs/")
SOURCE = candidates[-1]
DESTINATION = Path("/content/drive/MyDrive/dblock-checkpoint.ckpt")
DESTINATION.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(SOURCE, DESTINATION)
print(f"Copied {SOURCE} to {DESTINATION}")

## Compare samplers with the same checkpoint

Run both cells with the same checkpoint, `NUM_BLOCKS`, batch size, and inference-step count. The code defaults to `num_blocks` inference steps unless you pass `--num_inference_steps`. At equal steps, Heun makes roughly twice the denoiser calls per interval as Euler, plus the final denoising call. This first comparison checks quality at equal steps; a later comparison should reduce Heun steps and compare accuracy at a similar function-evaluation budget.

In [ ]:
CHECKPOINT = "/content/drive/MyDrive/dblock-checkpoint.ckpt"
INFERENCE_STEPS = NUM_BLOCKS
EVAL_BATCH_SIZE = 128

In [ ]:
!python main.py test cifar100 --model_type dblock \
  --ckpt_path "{CHECKPOINT}" --num_blocks {NUM_BLOCKS} \
  --num_inference_steps {INFERENCE_STEPS} --eval_batch_size {EVAL_BATCH_SIZE} \
  --num_workers 2 --ode euler

In [ ]:
!python main.py test cifar100 --model_type dblock \
  --ckpt_path "{CHECKPOINT}" --num_blocks {NUM_BLOCKS} \
  --num_inference_steps {INFERENCE_STEPS} --eval_batch_size {EVAL_BATCH_SIZE} \
  --num_workers 2 --ode heun

Record the reported test accuracy, inference-step count, and runtime for each method. The Heun path in this branch is experimental; verify its implementation before drawing conclusions from the comparison.